<a href="https://colab.research.google.com/github/Ronald-Tuncar/LAB07/blob/develop/LAB07_Ronald_Tuncar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalamos la librería para calcular el IV
!pip install -q pandas numpy scikit-learn statsmodels


In [ ]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.svm import SVC
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import pandas as pd

def get_data_from_url(url, column_names):
    '''
    Retrieve and prepare data from a CSV into a cleaned DataFrame.
    '''
    df = pd.read_csv(url, names=column_names)
    df['Bare_Nuclei'] = pd.to_numeric(df['Bare_Nuclei'], errors='coerce')
    df.dropna(inplace=True)
    df['Class'] = df['Class'].map({2: 0, 4: 1})
    df.drop('ID', axis=1, inplace=True)
    return df

# URL y nombres de columnas
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
column_names = ['ID', 'Clump_Thickness', 'Uniformity_Cell_Size', 'Uniformity_Cell_Shape',
                'Marginal_Adhesion', 'Single_Epithelial_Cell_Size', 'Bare_Nuclei',
                'Bland_Chromatin', 'Normal_Nucleoli', 'Mitoses', 'Class']

# Carga y limpieza
df = get_data_from_url(url, column_names)
df


,Clump_Thickness,Uniformity_Cell_Size,Uniformity_Cell_Shape,Marginal_Adhesion,Single_Epithelial_Cell_Size,Bare_Nuclei,Bland_Chromatin,Normal_Nucleoli,Mitoses,Class
0,5,1,1,1,2,1.0,3,1,1,0
1,5,4,4,5,7,10.0,3,2,1,0
2,3,1,1,1,2,2.0,3,1,1,0
3,6,8,8,1,3,4.0,3,7,1,0
4,4,1,1,3,2,1.0,3,1,1,0
...,...,...,...,...,...,...,...,...,...,...
694,3,1,1,1,3,2.0,1,1,1,0
695,2,1,1,1,2,1.0,1,1,1,0
696,5,10,10,3,7,3.0,8,10,2,1
697,4,8,6,4,3,4.0,10,6,1,1


**2. Calcular el Information Value (IV)**

In [ ]:
# Función para calcular el WOE y IV
def calc_iv(df, feature, target, bins=10):
    df = df[[feature, target]].copy()
    if pd.api.types.is_numeric_dtype(df[feature]):
        df['bin'] = pd.qcut(df[feature], q=bins, duplicates='drop')
    else:
        df['bin'] = df[feature]

    grouped = df.groupby('bin')[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    grouped['dist_good'] = grouped['good'] / grouped['good'].sum()
    grouped['dist_bad'] = grouped['bad'] / grouped['bad'].sum()
    grouped['woe'] = np.log((grouped['dist_good'] + 1e-6) / (grouped['dist_bad'] + 1e-6))
    grouped['iv'] = (grouped['dist_good'] - grouped['dist_bad']) * grouped['woe']
    iv = grouped['iv'].sum()
    return iv


In [ ]:
iv_dict = {}
target = 'Class'

for col in df.columns:
    if col != target:
        iv = calc_iv(df, col, target)
        iv_dict[col] = iv

iv_df = pd.DataFrame.from_dict(iv_dict, orient='index', columns=['IV']).sort_values(by='IV', ascending=False)
iv_df


<ipython-input-9-df5db5fb478b>:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('bin')[target].agg(['count', 'sum'])
<ipython-input-9-df5db5fb478b>:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('bin')[target].agg(['count', 'sum'])
<ipython-input-9-df5db5fb478b>:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = df.groupby('bin')[target].agg(['count', 'sum'])
<ipython-i

,IV
Uniformity_Cell_Size,9.722666
Uniformity_Cell_Shape,6.704774
Bland_Chromatin,6.417414
Clump_Thickness,5.743087
Normal_Nucleoli,4.923690
Bare_Nuclei,4.673082
Single_Epithelial_Cell_Size,4.089891
Marginal_Adhesion,3.343685
Mitoses,0.720707


In [ ]:
selected_features = iv_df[iv_df['IV'] >= 0.02].index.tolist()
X = df[selected_features]
y = df[target]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


# **4. Modelo de regresión logística y métricas**

In [ ]:
X_train_const = sm.add_constant(X_train)
logit_model = sm.Logit(y_train, X_train_const)
result = logit_model.fit()
print(result.summary())


Optimization terminated successfully.
         Current function value: 0.073760
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                  Class   No. Observations:                  512
Model:                          Logit   Df Residuals:                      502
Method:                           MLE   Df Model:                            9
Date:                Thu, 01 May 2025   Pseudo R-squ.:                  0.8842
Time:                        00:43:56   Log-Likelihood:                -37.765
converged:                       True   LL-Null:                       -326.13
Covariance Type:            nonrobust   LLR p-value:                2.070e-118
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const                         -10.0640      1.408     -7.150      0.000   

In [ ]:
# Filtramos variables con p-valor < 0.05
significant_vars = result.pvalues[result.pvalues < 0.05].index.tolist()
significant_vars.remove('const')


In [ ]:
log_model = LogisticRegression()
log_model.fit(X_train[significant_vars], y_train)
y_pred_log = log_model.predict(X_test[significant_vars])

# Métricas
print("Accuracy:", metrics.accuracy_score(y_test, y_pred_log))
print("Precision:", metrics.precision_score(y_test, y_pred_log))
print("Recall:", metrics.recall_score(y_test, y_pred_log))
print("F1-score:", metrics.f1_score(y_test, y_pred_log))
print("AUC:", metrics.roc_auc_score(y_test, y_pred_log))


Accuracy: 0.9590643274853801
Precision: 0.9841269841269841
Recall: 0.9117647058823529
F1-score: 0.9465648854961832
AUC: 0.9510279840091376


###**5. Modelo SVM y comparación**

In [ ]:
svm_model = SVC(kernel='linear', probability=True)
svm_model.fit(X_train[significant_vars], y_train)
y_pred_svm = svm_model.predict(X_test[significant_vars])

# Métricas
print("SVM Accuracy:", metrics.accuracy_score(y_test, y_pred_svm))
print("SVM Precision:", metrics.precision_score(y_test, y_pred_svm))
print("SVM Recall:", metrics.recall_score(y_test, y_pred_svm))
print("SVM F1-score:", metrics.f1_score(y_test, y_pred_svm))
print("SVM AUC:", metrics.roc_auc_score(y_test, y_pred_svm))


SVM Accuracy: 0.9473684210526315
SVM Precision: 0.9538461538461539
SVM Recall: 0.9117647058823529
SVM F1-score: 0.9323308270676691
SVM AUC: 0.9413192461450599


# **6. Comparación final**

In [ ]:
models = ['Logistic Regression', 'SVM']
metrics_data = {
    'Accuracy': [metrics.accuracy_score(y_test, y_pred_log), metrics.accuracy_score(y_test, y_pred_svm)],
    'Precision': [metrics.precision_score(y_test, y_pred_log), metrics.precision_score(y_test, y_pred_svm)],
    'Recall': [metrics.recall_score(y_test, y_pred_log), metrics.recall_score(y_test, y_pred_svm)],
    'F1 Score': [metrics.f1_score(y_test, y_pred_log), metrics.f1_score(y_test, y_pred_svm)],
    'AUC': [metrics.roc_auc_score(y_test, y_pred_log), metrics.roc_auc_score(y_test, y_pred_svm)],
}
results_df = pd.DataFrame(metrics_data, index=models)
results_df


,Accuracy,Precision,Recall,F1 Score,AUC
Logistic Regression,0.959064,0.984127,0.911765,0.946565,0.951028
SVM,0.947368,0.953846,0.911765,0.932331,0.941319
